In [ ]:
%run src/donnees.py

# df_caract_recoder
# df_lieux_recoder
# df_vehicule_recoder
# df_usager_recoder
# df_final

# Modelisation :
- classification avec gravité en variables catégorielle : 1 = Indemne, 2 = Blessé léger, 3 = Blessé hospitalisé, 4 = Tué
- Random Forest et/ou Regression logistique
- Facteurs clés : type de véhicule, ancienneté du véhicule, choc avant / arrière / latéral, âge, sexe, port de la ceinture / casque, alcoolémie, météo, luminosité, état de la chaussée, type de route (autoroute, nationale, départementale), zone (urbaine / rurale), département, zone accidentogène (a rajouter ?), heure, saison (a rajouter ?)
- A faire : one hot encoding, gérer les valeurs manquantes 

In [ ]:
df_final["grav"].unique()

In [ ]:
df_final.columns

## I - Random Forest

In [ ]:
grav_dict = {
    "Indemne":1,
    "Tué":4,
    "Blessé hospitalisé":3,
    "Blessé léger":2
}
df_final = recodage(df_final, {"grav": grav_dict})
df_final = df_final.dropna(subset=['grav'])

In [ ]:
df_final.info()

In [ ]:
y = df_final["grav"].astype(int).astype(str)
X = df_final[[
    "mois", "an", "lum", "dep", "circ", "surf", "vma", "catv", "obs",
    "obsm", "choc", "sexe", "trajet", "secu1", "age"
]]
X["an"] = X["an"].astype(str)

### Vérification des valeurs manquantes - A FAIRE

In [ ]:
X.info()

### Encodage des variables explicatives 

In [ ]:
import pandas as pd

X_encoded = pd.get_dummies(X, drop_first=True)

# Train, test 
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.2, random_state=66, stratify=y
)
# startify pour les classes déséquilibré 


In [ ]:
X_train

### Entrainement du modèle 

A faire : 
- optimisation des paramètres 
- penalisation en fonction des classes (pas bcp de tué)
- gestion des non renseigné et -1


In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    min_samples_split=5,
    min_samples_leaf=2,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)


### Evaluation et importance des facteurs 

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

y_pred = rf.predict(X_test)

print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))




In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# y_true = vraies classes
# y_pred = classes prédites

cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Purples',
            xticklabels=['Prédit 1', 'Prédit 2','Prédit 3', 'Prédit 4'],
            yticklabels=['Réel 1', 'Réel 2', 'Réel 3', 'Réel 4'])
plt.xlabel('Prédictions')
plt.ylabel('Réel')
plt.title('Matrice de confusion')
plt.show()


In [ ]:
import numpy as np

importances = rf.feature_importances_
indices = np.argsort(importances)[::-1]

for i in indices[:20]:
    print(f"{X_encoded.columns[i]} : {importances[i]:.4f}")

In [ ]:
import numpy as np

cm = confusion_matrix(y_test, y_pred)
cm_norm = cm / np.sum(cm)

plt.figure(figsize=(6,4))
sns.heatmap(cm_norm, annot=True, fmt='.2%', cmap='Purples',
            xticklabels=['Prédit 1', 'Prédit 2','Prédit 3', 'Prédit 4'],
            yticklabels=['Réel 1', 'Réel 2', 'Réel 3', 'Réel 4'])
plt.xlabel('Prédictions')
plt.ylabel('Réel')
plt.title('Matrice de confusion normalisée')
plt.show()


les classes 1 et 2 sont bien prédites, pour les classes 3 et 4 bcp moins debonnes prédictions, cela est du au fait que les données comporte moins d'accidnet comportant ces gravités de blessures, le 3 et 4 sont tué et blessures hospitalisé.
Sauf que nous on voudrait notamment prédire les tués pour voir ce qui est le plus dangereux ! Il va falloir donc rajouter des pénalisation :